# Imports

In [1]:
import cv2
import numpy as np
import mediapipe as mp
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

# Functions

In [2]:
def calculate_angle(a, b, c):
    a = np.array(a) # Start
    b = np.array(b) # Middle
    c = np.array(c) # End

    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians*180.0/np.pi)

    if angle > 180.0:
        angle = 360 - angle
    
    return angle

# Squat Analyzer Prototype

<img src="https://i.imgur.com/3j8BPdc.png" style="height:500px">

In [3]:
# Render detections
relevant_landmarks = [
    mp_pose.PoseLandmark.LEFT_SHOULDER,
    mp_pose.PoseLandmark.RIGHT_SHOULDER,
    mp_pose.PoseLandmark.LEFT_ELBOW,
    mp_pose.PoseLandmark.RIGHT_ELBOW,
    mp_pose.PoseLandmark.LEFT_WRIST,
    mp_pose.PoseLandmark.RIGHT_WRIST,
    mp_pose.PoseLandmark.LEFT_HIP,
    mp_pose.PoseLandmark.RIGHT_HIP,
    mp_pose.PoseLandmark.LEFT_KNEE,
    mp_pose.PoseLandmark.RIGHT_KNEE,
    mp_pose.PoseLandmark.LEFT_ANKLE,
    mp_pose.PoseLandmark.RIGHT_ANKLE,
    mp_pose.PoseLandmark.LEFT_HEEL,
    mp_pose.PoseLandmark.RIGHT_HEEL,
    mp_pose.PoseLandmark.LEFT_FOOT_INDEX,
    mp_pose.PoseLandmark.RIGHT_FOOT_INDEX
]

landmark_specs = {}

for landmark in mp_pose.PoseLandmark:
    if landmark in relevant_landmarks:
        landmark_specs[landmark] = mp_drawing.DrawingSpec(color = (0, 255, 0), thickness = 2, circle_radius = 2)
    else:
        landmark_specs[landmark] = mp_drawing.DrawingSpec(color = (0, 0, 0), thickness = 1, circle_radius = 1)

connection_specs = {}

for connection in mp_pose.POSE_CONNECTIONS:
    start_landmark = connection[0]
    end_landmark = connection[1]
    if start_landmark in relevant_landmarks and end_landmark in relevant_landmarks:
        connection_specs[connection] = mp_drawing.DrawingSpec(color=(255, 0, 0), thickness = 2)
    else:
        connection_specs[connection] = mp_drawing.DrawingSpec(color=(0, 0, 0), thickness = 1)

In [ ]:
# VIDEO FEED
cap = cv2.VideoCapture('video.mp4')
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
out = cv2.VideoWriter('output.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

# Rep information
rep_count = 0
hit_depth = False
eccentric = False
depth_status = "ECCENTRIC"
prev_hip_y = 0

# Mediapipe instance
with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.3) as pose:
    while cap.isOpened():
        ret, frame = cap.read()
        frame = cv2.resize(frame, (1280, 720))

        if not ret:
            break

        # Change colors from BGR to RGB
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False #Saves Memory

        # Make detection
        results = pose.process(image)

        # Change back to BGR
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        # Extract landmarks
        try:
            landmarks = results.pose_landmarks.landmark

            # Get coordinates of hip, shoulder, and knee landmarks
            shoulder = [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y]
            hip = [landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y]
            knee = [landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y]

            # Hip angle
            angle = calculate_angle(shoulder, hip, knee)
            cv2.putText(image, str(int(angle)),
                        tuple(np.multiply(hip, [width, height]).astype(int)),
                        cv2.FONT_HERSHEY_COMPLEX, 0.5, (255, 255, 255), 2, cv2.LINE_AA)
            
            # Rep and depth tracking
            current_hip_y = hip[1]
            threshold = 0.005

            if current_hip_y > prev_hip_y + threshold:
                if not eccentric:
                    hit_depth = False
                    depth_status = "ECCENTRIC"
                eccentric = True
            
            if eccentric and hip[1] > knee[1]:
                hit_depth = True
                depth_status = "DEPTH"
            
            if eccentric and current_hip_y < prev_hip_y - threshold:
                eccentric = False
                if hit_depth:
                    rep_count += 1
                else:
                    depth_status = "NO DEPTH"

            prev_hip_y = current_hip_y

            # Display rep count and depth indicator
            depth_color = (0, 255, 255) if depth_status == "ECCENTRIC" else (0, 255, 0) if depth_status == "DEPTH" else (0, 0, 255)
            
            cv2.putText(image, f'Reps: {rep_count}', (50, 50),
                        cv2.FONT_HERSHEY_COMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)
            cv2.putText(image, depth_status, (50, 100),
                        cv2.FONT_HERSHEY_COMPLEX, 1, depth_color, 2, cv2.LINE_AA)
            
            #print(f"hip_y: {hip[1]:.4f}, knee_y: {knee[1]:.4f}, eccentric: {eccentric}, hit_depth: {hit_depth}, depth_status: {depth_status}")

        except:
            pass

        # Render detections
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                                  landmark_drawing_spec = landmark_specs,
                                  connection_drawing_spec = connection_specs)

        out.write(image)

    cap.release()
    out.release()
    cv2.destroyAllWindows()

I0000 00:00:1776283165.472677   61745 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1776283165.597165  115060 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.8-0ubuntu0.24.04.1), renderer: llvmpipe (LLVM 20.1.2, 256 bits)
W0000 00:00:1776283166.080610  115057 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776283166.301858  115059 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
